In [55]:
from Run_Pipeline.Agent_Tool import AgentTools
from Run_Pipeline.Document_Loader import DocumentLoader
from Run_Pipeline.Document_Splitter import DocumentSplitter
from Run_Pipeline.Embedding_DB import EmbeddingDB
from Run_Pipeline.Env_Loader import (EnvLoader)
from Run_Pipeline.LLM_Factory import LLMFactory
from Run_Pipeline.Retriever_Builder import RetrieverBuilder
from pathlib import Path
import os

EnvLoader.load_local(dotenv_path="../.env")

BASE_DIR = Path(os.getcwd()).parent

kind = "json"
file_path = BASE_DIR / os.getenv("DATA_PATH")
chunk_size = 512
overlap_size = 50
device = "mps"
persist_dir = BASE_DIR / os.getenv("DATA_PATH") / f"{kind}_{chunk_size}"
retriever_mode = 3
k = 3
engine_num = 1
backend_num = 1



In [56]:
def build_chain():
    """
    RAG 파이프라인을 초기화해 LangSmith가 매 예제마다 새로 씁니다.
    """
    # --- ☑️ 당신이 앞서 작성한 코드 재사용 ---
    loader = DocumentLoader(file_path, kind)
    docs_post, docs_company = loader._load_json_files()

    splitter = DocumentSplitter(chunk_size=512, overlap=50)
    chunks = splitter.split(docs_post + docs_company, cache_dir=file_path)

    embed_db = EmbeddingDB(
        model_name="nlpai-lab/KURE-v1",
        device="mps",
        persist_dir=persist_dir
    )
    db = embed_db.get_or_create_db(chunks)

    retriever = RetrieverBuilder(
        db=db,
        full_docs=docs_post + docs_company,
        chunks=chunks,
        k=3,
    ).build(mode = retriever_mode)

    llm = LLMFactory.load_llm(engine=1, backend=1, device="mps")

    from Run_Pipeline.Chain_Factory import ChainFactory
    return ChainFactory.build_chain(retriever=retriever, llm=llm)


In [57]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from Run_Pipeline.LLM_Factory import LLMFactory
# from Run_Pipeline.Retriever_Builder import RetrieverBuilder
# from Run_Pipeline.Embedding_DB import EmbeddingDB
# from Run_Pipeline.Document_Splitter import DocumentSplitter
# from Run_Pipeline.Document_Loader import DocumentLoader
#
#
# def _len_okt(self, text: str) -> int:
#     return len(self.okt.morphs(text))
#
#
# def _okt_tokenize(self, text: str):
#     return self.okt.morphs(text)
#
#
# def build_chain():
#     """
#     RAG 파이프라인을 초기화해 LangSmith가 매 예제마다 새로 씁니다.
#     """
#     # --- ☑️ 당신이 앞서 작성한 코드 재사용 ---
#     loader = DocumentLoader(file_path, kind)
#     docs_post, docs_company = loader._load_json_files_simple()
#
#     splitter = DocumentSplitter(chunk_size=chunk_size, overlap=overlap_size)  # 이미 분할된 문서가 있다면 로드하고 아니면 새로 스플릿함
#     chunks = splitter.split_simple(docs_post + docs_company, cache_dir=file_path)  # 게시글과 회사 정보를 합쳐서 분할
#
#
#     embed_db = EmbeddingDB(
#         model_name="nlpai-lab/KURE-v1",
#         device="mps",
#         persist_dir=persist_dir
#     )
#     db = embed_db.get_or_create_db_simple(chunks)
#
#     retriever = RetrieverBuilder(
#         db=db,
#         full_docs=docs_post + docs_company,
#         chunks=chunks,
#         k=3,
#     ).build(mode=retriever_mode)
#
#     llm = LLMFactory.load_llm(engine=1, backend=1, device="mps")
#
#     from Run_Pipeline.Chain_Factory import ChainFactory
#     return ChainFactory.build_chain(retriever=retriever, llm=llm)


In [ ]:
chain = build_chain()

In [58]:
import pandas as pd

df = pd.read_excel('../Data_Files/RAG_test_questions.xlsx')
my_questions = df['Question'].to_list()

In [60]:
# rag_to_langsmith.py
"""질문 리스트를 RAG 체인에 통과시켜 (질문, 답변, 맥락)을 LangSmith 데이터셋에 업로드하는 스크립트.

사용법 예시:

    from your_build_module import build_chain  # 기존 build_chain 가져오기
    from rag_to_langsmith import upload_qac_dataset

    questions = [
        "회사의 핵심 사업은 무엇인가요?",
        "재택근무 제도는 있나요?",
        # ...
    ]

    upload_qac_dataset(build_chain, questions, dataset_name="rag_qac_dataset")
"""

from __future__ import annotations

import json
from typing import Callable, Dict, List, Any

from langsmith import Client
from langchain.schema import Document


###############################################################################
# LangSmith 업로드 유틸
###############################################################################

def _upload_to_langsmith(records: List[Dict[str, str]], dataset_name: str) -> str:
    """records = [{question, answer, context}, ...] 형태를 LangSmith에 업로드."""
    client = Client()

    # 기존 동일 이름 데이터셋이 있으면 삭제
    for ds in client.list_datasets():
        if ds.name == dataset_name:
            client.delete_dataset(dataset_id=ds.id)
            break

    dataset = client.create_dataset(dataset_name=dataset_name)

    for idx, r in enumerate(records, 1):
        client.create_example(
            dataset_id=dataset.id,
            inputs={"question": r["question"]},
            outputs={
                "answer": r["answer"],
                "context": r["context"],
            },
            metadata={"row": idx},
        )

    print(f"[LangSmith] '{dataset_name}' 업로드 완료! (총 {len(records)}개)")
    return dataset_name


###############################################################################
# RAG 체인 실행 결과에서 answer / context 추출 유틸
###############################################################################


def _extract_answer(result: Any) -> str:
    """체인 invoke 결과에서 답변 문자열 추출."""
    if isinstance(result, str):
        return result
    if isinstance(result, dict):
        # 수정된 ChainFactory의 결과 구조
        if "answer" in result:
            return str(result["answer"])
        # 기존 구조들
        for key in ("result", "output"):
            if key in result:
                return str(result[key])
    # LangChain Runnable 시나리오: LLMResult list 등
    return str(result)


def _extract_context(result: Any) -> str:
    """체인 invoke 결과에서 맥락(source_documents) 텍스트 추출."""
    # 수정된 ChainFactory의 결과 구조
    if isinstance(result, dict):
        if "context" in result:
            return str(result["context"])
        if "source_documents" in result:
            docs = result["source_documents"]
            if hasattr(docs, '__iter__'):
                return "\n\n".join(getattr(doc, 'page_content', str(doc)) for doc in docs)

    # 1) attr 로 존재할 때
    if hasattr(result, "source_documents"):
        docs = getattr(result, "source_documents")
        return "\n\n".join(doc.page_content for doc in docs)

    # 2) dict 구조 - 기존
    if isinstance(result, dict):
        if "retrieved_docs" in result:
            return "\n\n".join(str(d) for d in result["retrieved_docs"])

    return "(맥락 추출 불가)"


###############################################################################
# 디버깅 유틸리티
###############################################################################

def debug_chain_input(build_chain_fn: Callable[[], Any]) -> None:
    """체인의 입력 스키마를 확인하는 디버깅 함수"""
    chain = build_chain_fn()
    print("=" * 50)
    print("Chain Debug Information")
    print("=" * 50)
    print(f"Chain type: {type(chain)}")

    # 입력 스키마 확인
    try:
        if hasattr(chain, 'input_schema'):
            schema = chain.input_schema.schema()
            print(f"Input schema: {schema}")
            if 'properties' in schema:
                print(f"Expected input keys: {list(schema['properties'].keys())}")
    except Exception as e:
        print(f"Cannot get input schema: {e}")

    # 체인 구조 확인
    if hasattr(chain, 'steps'):
        print(f"Chain steps: {[type(step).__name__ for step in chain.steps]}")

    print("=" * 50)


###############################################################################
# 메인 함수
###############################################################################


def upload_qac_dataset(
        build_chain_fn: Callable[[], Any],
        questions: List[str],
        dataset_name: str = "rag_qac_dataset",
        show_progress: bool = True,
) -> str:
    """build_chain 함수와 질문 리스트를 받아 LangSmith 데이터셋 생성.

    Args:
        build_chain_fn: 빌드된 Chain 을 반환하는 콜러블 (ex. build_chain)
        questions: 문자열 질문 리스트
        dataset_name: LangSmith 상에 생성할 데이터셋 이름
        show_progress: True면 진행 상황 print
    Returns:
        dataset_name (str)
    """
    chain = build_chain_fn()
    records: List[Dict[str, str]] = []
    for idx, q in enumerate(questions, 1):
        if show_progress:
            print(f"[{idx}/{len(questions)}] Q: {q[:30]}…", end=" ")
        try:
            # ChainFactory.build_chain은 "input" 키를 기대함
            result = chain.invoke({"input": q})
            answer = _extract_answer(result)
            context = _extract_context(result)

        except Exception as e:
            answer = f"오류 발생: {e}"
            context = ""
        records.append({"question": q, "answer": answer, "context": context})
        if show_progress:
            print("✅")

    return _upload_to_langsmith(records, dataset_name)

In [61]:
if __name__ == "__main__":

    # 데이터셋 업로드 실행
    upload_qac_dataset(
        build_chain,
        my_questions,
        dataset_name="rag_qac_dataset_Retriever_3",  # ↔︎ 실제 이름
        show_progress=True
    )

캐시에서 62865개의 문서 청크 로드 완료
▶ 기존 Chroma DB 로드 중...
▶ 로드 완료
▶ BM25 색인 중…


BM25 문서 인덱싱: 100%|██████████| 62865/62865 [00:00<00:00, 1377712.69doc/s]


[1/30] Q: 컬리 ‘프로모션 기획 마케터’ 포지션의 최소 필요 경력… 컬리의 '프로모션 기획 마케터' 포지션에 대한 정보는 제공되지 않았습니다. 그러나, 마케팅 관련 직무에서 일반적으로 요구되는 최소 경력은 5년 이상인 경우가 많습니다. 정확한 경력 요건은 컬리의 공식 채용 공고를 확인하시거나 인사팀에 문의하시는 것이 좋습니다.✅
[2/30] Q: 씨제이올리브영 ‘커머스디자이너’ 포지션 지원 시 필수로… 씨제이올리브영의 '커머스디자이너' 포지션에 지원할 때 필수로 제출해야 하는 것은 다음과 같습니다:

1. 이력서
2. 포트폴리오

이력서에는 희망 연봉을 반드시 기재해야 하며, 포트폴리오는 지원자의 디자인 역량을 보여줄 수 있는 중요한 자료입니다. 서류 제출 시 파일 크기가 50MB를 넘는 경우에는 링크를 첨부하여 제출해야 합니다.✅
[3/30] Q: 111퍼센트 ‘웹 프론트 개발자’ 포지션에서 주 사용 … 111퍼센트의 '웹 프론트 개발자' 포지션에서 주 사용 프레임워크는 React입니다. 이 포지션에서는 React와 TypeScript를 이용해 신규 어플리케이션을 개발하고, 사내 인하우스 툴의 프론트엔드 신규 개발 및 유지보수를 담당하게 됩니다.✅
[4/30] Q: 토스페이먼츠 ‘Frontend Developer’ 포지… 토스페이먼츠의 'Frontend Developer' 포지션에서 사용하는 상태 관리 라이브러리는 Riverpod, BLoC, Provider입니다. 이 라이브러리들은 Flutter 애플리케이션에서 상태 관리를 효율적으로 수행하는 데 도움을 줍니다.✅
[5/30] Q: 미리디 ‘미리캔버스 프론트엔드팀’이 모노레포 구현을 위… 미리디 '미리캔버스 프론트엔드팀'이 모노레포 구현을 위해 도입한 도구는 **Turborepo**입니다. Turborepo를 활용하여 멀티 패키지 관리 및 빌드 속도를 최적화하고, CICD 환경에서도 효율적인 모듈 배포를 가능하게 하여 개발 및 배포 속도를 극대화하고 있습니다.✅
[6/30] Q: 여기어때 ‘Produ

# LangSmith 데이터셋에서 가져오기 이름만 바꾸면 됨

In [36]:
# 1. LangSmith 클라이언트 준비
from langsmith import Client
import pandas as pd  # 표로 보고 싶다면

client = Client()

# 2. 데이터셋 객체 찾기  ───────────────────────────────
DATASET_NAME = "rag_qac_dataset_simple_chunk_Retriever_1"  # ↔︎ 실제 이름
ds = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)

# 3. 예제(샘플) 목록 가져오기
examples = client.list_examples(dataset_id=ds.id)

# 4. 원하는 컬럼만 추려서 확인
records = []
for ex in examples:
    q = ex.inputs.get("question", "")
    a = ex.outputs.get("answer", "")
    ctx = ex.outputs.get("context", "")  # 또는 ex.outputs["context"]
    records.append({"question": q, "context": ctx, "answer": a})

# 5-a. 터미널에서 빠르게 보기
for r in records[:3]:
    print("Q:", r["question"])
    print("A:", r["answer"][:80], "…")
    print("CTX:", r["context"][:80], "…")
    print("-" * 40)

# 5-b. 판다스 데이터프레임으로
df = pd.DataFrame(records)
df



Q: 미리디 ‘미리캔버스 프론트엔드팀’이 Turborepo를 CICD 환경에서 활용함으로써 모듈 배포 효율화가 개발 및 배포 속도에 주는 영향은 무엇이라고 설명하는가?
A: 미리디의 미리캔버스 프론트엔드팀이 Turborepo를 CICD 환경에서 활용함으로써 모듈 배포 효율화를 이루는 것은 여러 가지 긍정적인 영향을  …
CTX: 게시글 ID: 282656
제목: 미리캔버스 프론트엔드 개발자
포지션 상세 설명: 미리캔버스 프론트엔드팀이 해결하려는 문제 빠르게 성장하는 서비 …
----------------------------------------
Q: 씨제이올리브영 ‘UI/UX 디자이너(신사업 프로덕트)’ 자격 요건에서 ‘정량적·정성적 데이터 기반 개선 경험’은 어느 범주에 속하며, 필수 Experience 최소 경력은 몇 년인가?
A: 씨제이올리브영의 ‘UI/UX 디자이너(신사업 프로덕트)’ 자격 요건에서 ‘정량적·정성적 데이터 기반 개선 경험’은 필수 Experience 범주 …
CTX: 자격 요건: ['712년 이내 Front Application 개발 경력', ' javascripttypescript 중에 숙련된 개발 경험 보 …
----------------------------------------
Q: 강남언니 ‘프로덕트 디자이너 주니어’가 글로벌 디자이너로 성장할 수 있는 이유로 제시된 두 가지 근거는 무엇인가?
A: 강남언니 ‘프로덕트 디자이너 주니어’가 글로벌 디자이너로 성장할 수 있는 이유로 제시된 두 가지 근거는 다음과 같습니다:

1. **다양한 국가 …
CTX: 수 있어요 리서치나 실험을 직접 설계하여 고객이 원하는 것을 발견하고, 가설을 검증하는 솔루션에 집중해 고객과 가장 가까운 곳에서 디자이너로서  …
----------------------------------------


,question,context,answer
0,미리디 ‘미리캔버스 프론트엔드팀’이 Turborepo를 CICD 환경에서 활용함으로...,게시글 ID: 282656\n제목: 미리캔버스 프론트엔드 개발자\n포지션 상세 설명...,미리디의 미리캔버스 프론트엔드팀이 Turborepo를 CICD 환경에서 활용함으로써...
1,씨제이올리브영 ‘UI/UX 디자이너(신사업 프로덕트)’ 자격 요건에서 ‘정량적·정성...,"자격 요건: ['712년 이내 Front Application 개발 경력', ' j...",씨제이올리브영의 ‘UI/UX 디자이너(신사업 프로덕트)’ 자격 요건에서 ‘정량적·정...
2,강남언니 ‘프로덕트 디자이너 주니어’가 글로벌 디자이너로 성장할 수 있는 이유로 제...,"수 있어요 리서치나 실험을 직접 설계하여 고객이 원하는 것을 발견하고, 가설을 검증...",강남언니 ‘프로덕트 디자이너 주니어’가 글로벌 디자이너로 성장할 수 있는 이유로 제...
3,페이타랩 ‘패스오더 프론트엔드 개발자’ 개발환경에서 빌드·컨테이너 도구 두 가지와 ...,회사명: 페이타랩패스오더\n근무지: 서울\n지역: 강남구\n직군: 개발\n직무: [...,페이타랩 ‘패스오더 프론트엔드 개발자’의 개발환경에서 사용되는 빌드 도구는 **We...
4,타다 ‘프로덕트 디자이너’ 채용에서 직무 인터뷰 다음 단계인 컬쳐핏 인터뷰에는 어떤...,"복지: ['# 몰입을 위한 모든 것을 지원하는 타다 BOOST PACK ', '',...",타다의 '프로덕트 디자이너' 채용에서 컬쳐핏 인터뷰에는 주로 두 그룹의 참여자가 포...
5,여기어때 ‘Product Owner 파트너서비스’ 복지에서 연간 100만 포인트와 ...,"우대 사항: ['국내외 주요 이커머스 서비스의 백오피스셀러오피스 경험이 있는 분',...",여기어때컴퍼니의 'Product Owner 파트너서비스' 직무에서 제공되는 연간 1...
6,미리디 ‘미리캔버스 프론트엔드팀’이 Micro Frontends 아키텍처 구현에 활...,게시글 ID: 282656\n제목: 미리캔버스 프론트엔드 개발자\n포지션 상세 설명...,미리디의 미리캔버스 프론트엔드팀이 Micro Frontends 아키텍처 구현에 활용...
7,토스페이먼츠 Frontend Developer 포지션에서 ‘결제 위젯’과 ‘대시보드...,"주요 업무: ['지금까진 이런 일을 했어요', '누구나 쉽게 원하는 결제 경험을 만...",토스페이먼츠의 Frontend Developer 포지션에서 '결제 위젯'과 '대시보...
8,111퍼센트 ‘웹 프론트 개발자’ 공고에 따르면 사옥은 2025년 8월경 어디로 확...,채용절차: ['서류전형 과제전형필요시 1차 인터뷰직무 2차 인터뷰컬처핏 처우...,"111퍼센트의 '웹 프론트 개발자' 공고에 따르면, 사옥은 2025년 8월경 성수동..."
9,컬리 ‘프로모션 기획 마케터’가 월 단위 주요 프로모션 테마를 기획할 때 포함하는 ...,게시글 ID: 283170\n제목: 프로모션 기획 마케터\n포지션 상세 설명: 컬리...,컬리의 '프로모션 기획 마케터'가 월 단위 주요 프로모션 테마를 기획할 때 포함해야...


# LLM 평가기

In [50]:
import pandas as pd
import json
import time
import re
from typing import Dict, List, Optional
import google.generativeai as genai
import openai
from langsmith import Client


class RAGEvaluator:
    def __init__(self, llm_provider: str = "gemini", api_key: str = None):
        """
        RAG 평가기 초기화

        Args:
            llm_provider: "gemini" 또는 "perplexity"
            api_key: API 키
        """
        self.llm_provider = llm_provider.lower()

        if self.llm_provider == "gemini":
            genai.configure(api_key=api_key)
            self.model = genai.GenerativeModel('gemini-2.0-flash')
        elif self.llm_provider == "perplexity":
            self.client = openai.OpenAI(
                api_key=api_key,
                base_url="https://api.perplexity.ai"
            )
        else:
            raise ValueError("llm_provider는 'gemini' 또는 'perplexity'여야 합니다.")

    def create_evaluation_prompt(self, question: str, context: str, answer: str, metric_type: str) -> str:
        """평가 메트릭별 프롬프트 생성 (개선된 버전)"""

        base_instruction = f"""
다음 RAG 시스템의 성능을 평가해주세요.

**질문 (Question):**
{question}

**검색된 맥락 (Context):**
{context}

**생성된 답변 (Answer):**
{answer}

---
"""

        if metric_type == "faithfulness":
            specific_instruction = """
**평가 기준: 신뢰성 (Faithfulness)**
생성된 답변이 검색된 맥락 데이터에 얼마나 사실적으로 부합하는지 평가하세요.
- 10점: 답변의 모든 내용이 맥락에서 직접 뒷받침됨
- 8-9점: 답변의 대부분이 맥락에 부합하며, 약간의 추론이 있음
- 6-7점: 답변의 일부가 맥락에 부합하지만 일부 내용이 불분명
- 4-5점: 답변의 일부만 맥락에 부합하며 상당한 추측이 포함됨
- 2-3점: 답변이 맥락과 크게 다르거나 모순됨
- 0-1점: 답변이 맥락과 전혀 부합하지 않음
"""

        elif metric_type == "answer_relevance":
            specific_instruction = """
**평가 기준: 답변 관련성 (Answer Relevance)**
생성된 답변이 질문과 얼마나 관련성이 있고 적절한지 평가하세요.
- 10점: 질문에 완벽하게 답변하며 정확하고 완전함
- 8-9점: 질문에 잘 답변하지만 약간의 부족한 부분이 있음
- 6-7점: 질문에 부분적으로 답변하지만 핵심을 놓침
- 4-5점: 질문과 관련은 있지만 부적절하거나 불완전함
- 2-3점: 질문과 약간 관련이 있지만 대부분 부적절함
- 0-1점: 질문과 전혀 관련이 없음
"""

        elif metric_type == "context_relevance":
            specific_instruction = """
**평가 기준: 맥락 관련성 (Context Relevance)**
검색된 맥락 데이터가 질문과 얼마나 관련이 있고 유용한지 평가하세요.
- 10점: 맥락이 질문에 완벽하게 관련되고 답변에 필요한 모든 정보 포함
- 8-9점: 맥락이 질문에 매우 관련되며 대부분의 필요한 정보 포함
- 6-7점: 맥락이 질문에 관련되지만 일부 정보가 부족하거나 불필요함
- 4-5점: 맥락이 질문에 부분적으로만 관련됨
- 2-3점: 맥락이 질문과 약간만 관련됨
- 0-1점: 맥락이 질문과 전혀 관련이 없음
"""

        # 더 명확한 출력 형식 지시
        output_format = """
**중요: 다음 형식으로 정확히 응답하세요**

반드시 아래 JSON 형식으로만 응답하고, 다른 텍스트는 포함하지 마세요:

{"score": [0부터 10까지의 숫자], "reasoning": "평가 근거"}

예시:
{"score": 7, "reasoning": "답변이 대부분 정확하지만 일부 세부사항이 부족함"}
"""

        return base_instruction + specific_instruction + output_format

    def get_llm_response(self, prompt: str) -> str:
        """LLM에서 응답 받기"""
        try:
            if self.llm_provider == "gemini":
                response = self.model.generate_content(prompt)
                return response.text

            elif self.llm_provider == "perplexity":
                response = self.client.chat.completions.create(
                    model="llama-3.1-sonar-large-128k-online",  # 또는 다른 모델
                    messages=[
                        {"role": "system",
                         "content": "당신은 RAG 시스템 평가 전문가입니다. 정확하고 객관적인 평가를 수행하세요. 반드시 JSON 형식으로만 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.1
                )
                return response.choices[0].message.content

        except Exception as e:
            print(f"LLM 응답 오류: {e}")
            return None

    def parse_llm_response(self, response: str) -> Dict:
        """LLM 응답에서 점수와 근거 추출 (가장 견고한 버전)"""

        def clean_response(text: str) -> str:
            """응답 텍스트 정리"""
            text = text.strip()

            # Markdown 코드 블록 제거
            if text.startswith('```json'):
                text = text[7:]
            elif text.startswith('```'):
                text = text[3:]

            if text.endswith('```'):
                text = text[:-3]

            return text.strip()

        def extract_json_objects(text: str) -> List[str]:
            """텍스트에서 모든 JSON 객체 추출"""
            json_objects = []
            brace_count = 0
            start_pos = -1

            for i, char in enumerate(text):
                if char == '{':
                    if brace_count == 0:
                        start_pos = i
                    brace_count += 1
                elif char == '}':
                    brace_count -= 1
                    if brace_count == 0 and start_pos != -1:
                        json_objects.append(text[start_pos:i + 1])
                        start_pos = -1

            return json_objects

        def try_parse_json(json_str: str) -> Optional[Dict]:
            """JSON 파싱 시도"""
            try:
                return json.loads(json_str)
            except:
                return None

        try:
            # 1. 응답 정리
            clean_text = clean_response(response)

            # 2. JSON 객체들 추출
            json_candidates = extract_json_objects(clean_text)

            # 3. 각 JSON 객체 파싱 시도
            for json_str in json_candidates:
                result = try_parse_json(json_str)
                if result and 'score' in result:
                    # 점수 유효성 검사
                    score = result.get('score', 0)
                    if isinstance(score, (int, float)) and 0 <= score <= 10:
                        return {
                            'score': float(score),
                            'reasoning': result.get('reasoning', 'No reasoning provided')
                        }

            # 4. 정규표현식으로 점수와 이유 추출 (백업 방법)
            score_match = re.search(r'"score"\s*:\s*(\d+(?:\.\d+)?)', clean_text)
            reasoning_match = re.search(r'"reasoning"\s*:\s*"([^"]*)"', clean_text)

            if score_match:
                score = float(score_match.group(1))
                reasoning = reasoning_match.group(1) if reasoning_match else "추출된 점수"

                if 0 <= score <= 10:
                    return {
                        'score': score,
                        'reasoning': reasoning
                    }

            # 5. 모든 시도 실패
            raise ValueError("유효한 점수를 찾을 수 없습니다.")

        except Exception as e:
            print(f"파싱 오류: {e}")
            print(f"원본 응답: {response[:500]}...")  # 처음 500자만 출력
            return {'score': 0.0, 'reasoning': 'Parsing failed'}

    def evaluate_single_example(self, question: str, context: str, answer: str) -> Dict:
        """단일 예제에 대한 전체 평가"""
        metrics = ['faithfulness', 'answer_relevance', 'context_relevance']
        results = {}

        for metric in metrics:
            print(f"  평가 중: {metric}")

            prompt = self.create_evaluation_prompt(question, context, answer, metric)
            response = self.get_llm_response(prompt)

            if response:
                parsed = self.parse_llm_response(response)
                results[metric] = parsed
            else:
                results[metric] = {'score': 0.0, 'reasoning': 'LLM response failed'}

            # API 호출 간격 (rate limiting 방지)
            time.sleep(1)

        return results

    def evaluate_dataset(self, records: List[Dict]) -> pd.DataFrame:
        """전체 데이터셋 평가"""
        all_results = []

        for i, record in enumerate(records):
            print(f"\n평가 진행 중... ({i + 1}/{len(records)})")

            question = record['question']
            context = record['context']
            answer = record['answer']

            # 각 메트릭별 평가
            evaluation_results = self.evaluate_single_example(question, context, answer)

            # 결과 저장
            result_row = {
                'question': question,
                'context': context,
                'answer': answer,
                'faithfulness_score': evaluation_results['faithfulness']['score'],
                'faithfulness_reasoning': evaluation_results['faithfulness']['reasoning'],
                'answer_relevance_score': evaluation_results['answer_relevance']['score'],
                'answer_relevance_reasoning': evaluation_results['answer_relevance']['reasoning'],
                'context_relevance_score': evaluation_results['context_relevance']['score'],
                'context_relevance_reasoning': evaluation_results['context_relevance']['reasoning'],
                'average_score': (
                                         evaluation_results['faithfulness']['score'] +
                                         evaluation_results['answer_relevance']['score'] +
                                         evaluation_results['context_relevance']['score']
                                 ) / 3
            }

            all_results.append(result_row)

        return pd.DataFrame(all_results)

    def get_summary_statistics(self, df: pd.DataFrame) -> Dict:
        """평가 결과 요약 통계"""
        return {
            'faithfulness': {
                'mean': df['faithfulness_score'].mean(),
                'std': df['faithfulness_score'].std(),
                'min': df['faithfulness_score'].min(),
                'max': df['faithfulness_score'].max()
            },
            'answer_relevance': {
                'mean': df['answer_relevance_score'].mean(),
                'std': df['answer_relevance_score'].std(),
                'min': df['answer_relevance_score'].min(),
                'max': df['answer_relevance_score'].max()
            },
            'context_relevance': {
                'mean': df['context_relevance_score'].mean(),
                'std': df['context_relevance_score'].std(),
                'min': df['context_relevance_score'].min(),
                'max': df['context_relevance_score'].max()
            },
            'overall': {
                'mean': df['average_score'].mean(),
                'std': df['average_score'].std(),
                'min': df['average_score'].min(),
                'max': df['average_score'].max()
            }
        }



In [51]:
from Run_Pipeline.Env_Loader import EnvLoader

EnvLoader.load_local(dotenv_path="../.env")

In [53]:
import os


def main():
    # 1. LangSmith에서 데이터셋 가져오기 (기존 코드)
    client = Client()
    DATASET_NAME = "rag_qac_dataset_Retriever_4"  # 실제 데이터셋 이름으로 교체
    ds = next(ds for ds in client.list_datasets() if ds.name == DATASET_NAME)
    examples = client.list_examples(dataset_id=ds.id)

    records = []
    for ex in examples:
        q = ex.inputs.get("question", "")
        a = ex.outputs.get("answer", "")
        ctx = ex.outputs.get("context", "")
        records.append({"question": q, "context": ctx, "answer": a})

    # 2. 평가기 초기화
    API_KEY = os.getenv("PPLX_API_KEY")  # 실제 API 키로 교체

    # Gemini 사용하는 경우
    #evaluator = RAGEvaluator(llm_provider="gemini", api_key=API_KEY)

    # Perplexity 사용하는 경우
    evaluator = RAGEvaluator(llm_provider="perplexity", api_key=API_KEY)

    # 3. 평가 실행 (전체 데이터셋 또는 샘플)
    #sample_records = records[:5]  # 테스트용으로 5개만
    sample_records = records  # 전체 데이터셋 평가

    print("RAG 시스템 평가를 시작합니다...")
    results_df = evaluator.evaluate_dataset(sample_records)

    # 4. 결과 저장 및 요약

    # 요약 통계
    print("\n=== 평가 결과 요약 ===")
    print(f"총 평가 항목 수: {len(results_df)}")
    print(f"신뢰성 평균 점수: {results_df['faithfulness_score'].mean():.2f}")
    print(f"답변 관련성 평균 점수: {results_df['answer_relevance_score'].mean():.2f}")
    print(f"맥락 관련성 평균 점수: {results_df['context_relevance_score'].mean():.2f}")
    print(f"전체 평균 점수: {results_df['average_score'].mean():.2f}")

    # 상세 결과 출력
    print("\n=== 상세 결과 ===")
    print(
        results_df[['faithfulness_score', 'answer_relevance_score', 'context_relevance_score', 'average_score']].head())

    return results_df


if __name__ == "__main__":
    results = main()

RAG 시스템 평가를 시작합니다...

평가 진행 중... (1/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (2/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (3/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (4/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (5/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (6/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (7/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (8/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (9/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (10/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  평가 중: context_relevance

평가 진행 중... (11/30)
  평가 중: faithfulness
  평가 중: answer_relevance
  

In [28]:
results.head(32)

,question,context,answer,faithfulness_score,faithfulness_reasoning,answer_relevance_score,answer_relevance_reasoning,context_relevance_score,context_relevance_reasoning,average_score
0,영상화/멀티미디어 제작사 관련 공고 보여줘,게시글 ID: 268045\n제목: 제작 PD 광고 및 홍보 영상 콘텐츠 제작\n...,현재 영상화 및 멀티미디어 제작사와 관련된 공고는 다음과 같습니다:\n\n1. **...,9.0,"생성된 답변의 대부분이 검색된 맥락 데이터에 직접 뒷받침되며, 회사명, 근무지, 직...",9.0,"생성된 답변은 질문에 완벽하게対応하며, 필요한 정보를 정확하고 완전하게 제공합니다....",9.0,"검색된 맥락 데이터가 질문에 매우 관련되며, 필요한 모든 정보를 포함하고 있습니다....",9.000000
1,서울에 있는 GMO/식품/바이오 관련 회사 공고 정리해줘,회사명: 강남엄마\n\n회사명: 이스턴생명과학\n회사 소개: IT 업체\n업종(산업...,서울에 있는 GMO/식품/바이오 관련 회사의 공고를 정리하기 위해서는 해당 산업에 ...,8.0,"답변이 대부분 검색된 맥락에 부합하며, 지놈오피니언에 대한 상세한 정보를 제공함. ...",8.0,"답변이 질문과 관련되어 있으며, 지놈오피니언과 같은 바이오 관련 회사에 대한 상세한...",6.0,"답변이 질문에 관련되지만, 제공된 맥락 데이터에서 서울에 있는 GMO/식품/바이오 ...",7.333333
2,UX디자이너 채용 공고 좀 보여줘,"근무지: 서울\n지역: 강남구\n직군: 디자인\n직무: ['웹 디자이너', 'UX ...",현재 강남구에서 UX 디자이너 채용 공고는 다음과 같습니다:\n\n1. **게시글 ...,6.0,"답변이 일부 세부사항에서 맥락과 부합하지 않음. 예를 들어, 특정 회사나 상세한 채...",9.0,"생성된 답변은 질문에 완벽하게対応하며, 필요한 정보를 정확하고 완전하게 제공합니다....",9.0,"검색된 맥락 데이터가 질문에 매우 관련되며, 필요한 모든 정보를 포함하고 있습니다....",8.000000
3,디지털 마케팅 인턴 공고를 보여줘,게시글 ID: 268926\n제목: 인턴 한국 마케팅\n\n게시글 ID: 28944...,"현재 디지털 마케팅 인턴 공고를 직접 보여드릴 수는 없지만, 일반적으로 디지털 마케...",2.0,"답변이 맥락과 크게 다르며, 제공된 검색 결과와 직접적인 연결이 없음. 검색 결과에...",4.0,답변이 질문과 관련은 있지만 부적절하거나 불완전함. 실제 디지털 마케팅 인턴 공고의...,4.0,검색된 맥락 데이터가 디지털 마케팅 인턴 공고의具体적인 내용이나 자격 요건을 제공하...,3.333333
4,서울 금천구에 있는 PR회사 공고를 보여줘,회사명: 마인드리퍼블릭 existing\n회사 소개: 이커머스 PR 에이전시 리...,서울 금천구에 있는 PR 회사의 공고는 위더스애드에서 확인할 수 있습니다. 아래는 ...,9.0,"생성된 답변의 대부분이 검색된 맥락 데이터에 직접 뒷받침되며, 약간의 추론이 포함됨...",10.0,생성된 답변이 질문에 완벽하게 답변하며 정확하고 완전함. 질문에서 요청한 서울 금천...,9.0,"검색된 맥락 데이터가 질문에 매우 관련되며, 필요한 정보 대부분을 포함하고 있습니다...",9.333333
5,재택 근무를 할 수 있는 회사 공고 알려줘,"복지: ['재택근무제주1일 출근, 6개월 이상 재직자 대상', ' 시간연차 사용 지...",재택 근무를 지원하는 회사로는 다음과 같은 두 곳이 있습니다:\n\n1. **앨리슨...,9.0,"생성된 답변의 대부분이 검색된 맥락 데이터에 직접 뒷받침되며, 회사명, 근무지, 직...",9.0,"생성된 답변이 질문에 완벽하게対応하며, 재택 근무를 지원하는 회사들의 주요 정보를 ...",9.0,"검색된 맥락 데이터가 질문에 매우 관련되며, 재택 근무를 지원하는 회사들의 상세 정...",9.000000
6,고졸로 지원할 수 있는 대기업 공고를 알려줘,"특징: ['유급휴가', '커피스낵바', '식대지원', '설립10년이상', '연봉 업...","현재 제공된 정보에 따르면, 고졸로 지원할 수 있는 대기업 공고는 없습니다. 대부분...",2.0,"답변이 제공된 맥락과 크게 다르며, 모순된 내용이 포함됨. 제공된 데이터에는 고졸로...",4.0,"생성된 답변은 질문에 직접적으로 응답하지 못하고, 제공된 컨텍스트와 무관한 일반적인...",0.0,제공된 맥락 데이터가 질문과 전혀 관련이 없음. 질문은 고졸로 지원할 수 있는 대기...,2.000000
7,Growth Marketer직무 관련 공고 보여줘,게시글 ID: 285592\n제목: B2B SaaS Growth Marketing ...,현재 서울 성동구에서의 Growth Marketer 직무 관련 공고는 다음과 같습니...,9.0,"답변이 검색된 맥락 데이터에 대부분 사실적으로 부합하며, 약간의 추론이 포함됨. 게...",4.0,"생성된 답변은 실제로 검색된 자료와 일치하지 않으며, 제공된.context와도 맞지...",9.0,"검색된 맥락 데이터가 질문에 매우 관련되며, 필요한 대부분의 정보를 포함하고 있습니...",7.333333
8,프레인글로벌에 입사하고 싶은데 다른 나라에도 지사가 있는지 알고 싶어,회사명: 프레인글로벌\n근무지: 서울\n지역: 중구\n직군: 마케팅광고\n직무: [...,"프레인글로벌은 국내 1위 PR 기업으로, 해외 PR AWARDS에서 대상을 수상한 ...",6.0,"답변이 일부 맥락에 부합하지만, 프레인글로벌의海外 지사에 대한 정보가 명확히 제공되...",7.0,답변이 대부분 정확하지만 프레인글로벌의 해외 지사 여부에 대한 명확한 정보가 부족함...,4.0,검색된 맥락 데이터에 프레인글로벌의 해외 지사 또는 글로벌 사업에 대한 정보가 포함...,5.666667
9,KB국민은행에 어떤 직군이 있는지 알려줘,회사명: KB국민은행\n회사 소개: 금융을 통해 고객의 행복한 미래를 설계하고 보다...,"KB국민은행은 다양한 직군을 운영하고 있으며, 주요 직군은 다음과 같습니다:\n\n...",8.0,"답변이 대부분 검색된 맥락에 부합하지만, 일부 세부사항이 추가적으로 추론되거나 상세...",8.0,"답변이 대부분 정확하게 KB국민은행의 주요 직군을 설명했지만, 일부 세부적인 직무介...",9.0,"검색된 맥락 데이터가 질문에 매우 관련되어 있으며, 대부분의 필요한 정보를 포함하고...",8.333333


In [54]:
results.to_json('rag_evaluation_results_Retriever_4.json', orient='records', force_ascii=False, indent=4)